# NLP Assignment

**Name:** Krish
**Roll Number:** 2401730158
**Course:** B.Tech CSE – Artificial Intelligence and Machine Learning

## Questions
1. Implement a complete **Word2Vec + LSTM** text-classification pipeline.
2. Fine-tune a pretrained Transformer—**DistilBERT**—for text classification.

## Dataset used
A small subset of the **IMDb movie-review dataset** is used:

- **0 = Negative review**
- **1 = Positive review**



---
# Install and Import Required Libraries


In [ ]:
# Run this cell once. It installs the libraries needed for both questions.
%pip install -q gensim transformers datasets accelerate

print('All required packages are installed.')


In [ ]:
# Common libraries
import os
import re
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
import seaborn as sns

from gensim.models import Word2Vec
from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

# Set seed so that results are as reproducible as possible
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Use GPU if available, otherwise CPU
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Training device:', DEVICE)


**Expected output format**

```text
Training device: cuda       # or cpu
```

---
# QUESTION 1: WORD2VEC + LSTM CLASSIFICATION PIPELINE

## Step 1: Load a Small IMDb Dataset


In [ ]:
# We use a small subset so that training is quick.
# The IMDb labels are already available in the dataset.

train_data = load_dataset('imdb', split='train[:1000]')
valid_data = load_dataset('imdb', split='train[1000:1200]')
test_data = load_dataset('imdb', split='test[:200]')

train_texts = train_data['text']
valid_texts = valid_data['text']
test_texts = test_data['text']

train_labels = np.array(train_data['label'])
valid_labels = np.array(valid_data['label'])
test_labels = np.array(test_data['label'])

print('Training examples:', len(train_texts))
print('Validation examples:', len(valid_texts))
print('Testing examples:', len(test_texts))

print('\nLabel counts in training data:')
print(pd.Series(train_labels).value_counts().sort_index())

print('\nOne positive review:')
positive_index = int(np.where(train_labels == 1)[0][0])
print(train_texts[positive_index][:400])


**Expected output format**

```text
Training examples: 1000
Validation examples: 200
Testing examples: 200

Label counts in training data:
0    500
1    500
```

## Step 2: Clean the Text


In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'<br\s*/?>', ' ', text)
    text = re.sub(r'[^a-z]+', ' ', text)
    words = text.split()
    return words

# Apply the cleaning function
train_sentences = [clean_text(text) for text in train_texts]
valid_sentences = [clean_text(text) for text in valid_texts]
test_sentences = [clean_text(text) for text in test_texts]

print('Original text:')
print(train_texts[0][:300])

print('\nCleaned words:')
print(train_sentences[0][:30])


## Step 3: Train a Word2Vec Model


In [ ]:
EMBEDDING_SIZE = 50

word2vec_model = Word2Vec(
    sentences=train_sentences,
    vector_size=EMBEDDING_SIZE,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    epochs=5,
    seed=SEED,
)

print('Word2Vec vocabulary size:', len(word2vec_model.wv.index_to_key))
print('Vector shape for one word:', word2vec_model.wv['movie'].shape)

print('\nWords similar to "movie":')
for word, similarity in word2vec_model.wv.most_similar('movie', topn=5):
    print(f'{word:15s} {similarity:.3f}')


**Expected output format**

```text
Word2Vec vocabulary size: 10000+       # exact number may differ
Vector shape for one word: (50,)

Words similar to "movie":
film            0.75
movies          0.70
...
```

## Step 4: Create Word IDs and the Embedding Matrix


In [ ]:
word_to_id = {'<PAD>': 0, '<UNK>': 1}

for word in word2vec_model.wv.index_to_key:
    word_to_id[word] = len(word_to_id)

vocab_size = len(word_to_id)
print('Total vocabulary size used by LSTM:', vocab_size)

# Create a matrix where each row is the vector of one word
embedding_matrix = np.zeros((vocab_size, EMBEDDING_SIZE), dtype=np.float32)
embedding_matrix[1] = np.random.normal(0, 0.05, EMBEDDING_SIZE)

for word, word_id in word_to_id.items():
    if word in word2vec_model.wv:
        embedding_matrix[word_id] = word2vec_model.wv[word]

print('Embedding matrix shape:', embedding_matrix.shape)


**Expected output format**

```text
Total vocabulary size used by LSTM: 10000+
Embedding matrix shape: (10000+, 50)
```

## Step 5: Convert Reviews into Fixed-Length Sequences


In [ ]:
MAX_WORDS = 100

def make_sequence(words):
    sequence = [word_to_id.get(word, 1) for word in words[:MAX_WORDS]]
    sequence = sequence + [0] * (MAX_WORDS - len(sequence))
    return sequence

train_sequences = [make_sequence(words) for words in train_sentences]
valid_sequences = [make_sequence(words) for words in valid_sentences]
test_sequences = [make_sequence(words) for words in test_sentences]

print('Number of training sequences:', len(train_sequences))
print('Length of every sequence:', len(train_sequences[0]))
print('First 15 IDs of the first review:', train_sequences[0][:15])


## Step 6: Create PyTorch DataLoaders


In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = torch.tensor(sequences, dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.sequences[index], self.labels[index]


BATCH_SIZE = 32

train_loader = DataLoader(
    ReviewDataset(train_sequences, train_labels),
    batch_size=BATCH_SIZE,
    shuffle=True
)

valid_loader = DataLoader(
    ReviewDataset(valid_sequences, valid_labels),
    batch_size=BATCH_SIZE
)

test_loader = DataLoader(
    ReviewDataset(test_sequences, test_labels),
    batch_size=BATCH_SIZE
)

example_sequences, example_labels = next(iter(train_loader))
print('Batch shape:', example_sequences.shape)
print('Label batch shape:', example_labels.shape)


**Expected output format**

```text
Batch shape: torch.Size([32, 100])
Label batch shape: torch.Size([32])
```

## Step 7: Build the LSTM Model


In [ ]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_size, embedding_matrix):
        super().__init__()

        # Word vectors are loaded from the trained Word2Vec model
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_size,
            padding_idx=0
        )
        self.embedding.weight.data.copy_(
            torch.tensor(embedding_matrix, dtype=torch.float32)
        )

        # Bidirectional LSTM reads the sequence in both directions
        self.lstm = nn.LSTM(
            input_size=embedding_size,
            hidden_size=64,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.4)
        self.output_layer = nn.Linear(64 * 2, 1)

    def forward(self, sequence):
        embedded = self.embedding(sequence)
        lstm_output, (hidden, cell) = self.lstm(embedded)

        # Combine the final forward and backward hidden states
        final_hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        final_hidden = self.dropout(final_hidden)

        # One output value is used for binary classification
        output = self.output_layer(final_hidden).squeeze(1)
        return output


lstm_model = SentimentLSTM(
    vocab_size=vocab_size,
    embedding_size=EMBEDDING_SIZE,
    embedding_matrix=embedding_matrix
).to(DEVICE)

print(lstm_model)
print('Number of trainable parameters:',
      sum(parameter.numel() for parameter in lstm_model.parameters() if parameter.requires_grad))


## Step 8: Train the Word2Vec + LSTM Model


In [ ]:
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.001)
EPOCHS = 5

training_losses = []
validation_accuracies = []


def calculate_accuracy(model, data_loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for sequences, labels in data_loader:
            sequences = sequences.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(sequences)
            predictions = (torch.sigmoid(outputs) >= 0.5).float()

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return correct / total


for epoch in range(1, EPOCHS + 1):
    lstm_model.train()
    total_loss = 0

    for sequences, labels in train_loader:
        sequences = sequences.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = lstm_model(sequences)
        loss = loss_function(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)
    validation_accuracy = calculate_accuracy(lstm_model, valid_loader)

    training_losses.append(average_loss)
    validation_accuracies.append(validation_accuracy)

    print(f'Epoch {epoch}/{EPOCHS} | Loss: {average_loss:.4f} | Validation accuracy: {validation_accuracy:.4f}')


**Expected output format**

```text
Epoch 1/5 | Loss: 0.6500 | Validation accuracy: 0.6800
Epoch 2/5 | Loss: 0.5000 | Validation accuracy: 0.7600
...
```
Values may be different.

## Step 9: Show the Training Graph


In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(training_losses, marker='o', color='red')
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(validation_accuracies, marker='o', color='green')
plt.title('Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()


## Step 10: Test the Word2Vec + LSTM Model


In [ ]:
lstm_model.eval()
test_predictions = []
test_probabilities = []

with torch.no_grad():
    for sequences, labels in test_loader:
        sequences = sequences.to(DEVICE)
        outputs = lstm_model(sequences)

        probabilities = torch.sigmoid(outputs).cpu().numpy()
        predictions = (probabilities >= 0.5).astype(int)

        test_probabilities.extend(probabilities)
        test_predictions.extend(predictions)

test_predictions = np.array(test_predictions)
test_probabilities = np.array(test_probabilities)

lstm_accuracy = accuracy_score(test_labels, test_predictions)
lstm_f1 = f1_score(test_labels, test_predictions)

print('Word2Vec + LSTM TEST RESULTS')
print('Accuracy:', round(lstm_accuracy, 4))
print('F1-score:', round(lstm_f1, 4))
print('\nDetailed classification report:')
print(classification_report(
    test_labels,
    test_predictions,
    target_names=['Negative', 'Positive'],
    digits=4
))


## Step 11: Confusion Matrix for LSTM


In [ ]:
lstm_cm = confusion_matrix(test_labels, test_predictions)

plt.figure(figsize=(5, 4))
sns.heatmap(
    lstm_cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Negative', 'Positive'],
    yticklabels=['Negative', 'Positive']
)
plt.title('Word2Vec + LSTM Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.show()


## Step 12: Predict a New Review with LSTM


In [ ]:
def predict_with_lstm(review):
    words = clean_text(review)
    sequence = torch.tensor([make_sequence(words)], dtype=torch.long).to(DEVICE)

    lstm_model.eval()
    with torch.no_grad():
        output = lstm_model(sequence)
        probability = torch.sigmoid(output).item()

    predicted_class = 1 if probability >= 0.5 else 0
    predicted_name = 'Positive' if predicted_class == 1 else 'Negative'
    return predicted_name, probability


new_review = 'This movie was excellent. The acting was great and the story was very interesting.'
prediction, probability = predict_with_lstm(new_review)

print('Review:', new_review)
print('Predicted class:', prediction)
print('Positive probability:', round(probability, 4))


## Step 13: Save the LSTM Model


In [ ]:
os.makedirs('saved_models', exist_ok=True)
torch.save(lstm_model.state_dict(), 'saved_models/word2vec_lstm_model.pt')
word2vec_model.save('saved_models/word2vec_model.bin')

print('LSTM model saved as saved_models/word2vec_lstm_model.pt')
print('Word2Vec model saved as saved_models/word2vec_model.bin')


---
# QUESTION 2: FINE-TUNE DISTILBERT FOR TEXT CLASSIFICATION

DistilBERT is used because it is a smaller and faster version of BERT, but it still gives strong results.

## Step 1: Load DistilBERT Tokenizer and Model


In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments
)

MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 128

# tokenizer converts text into IDs understood by DistilBERT
bert_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# The classification model has 2 outputs: negative and positive
bert_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
).to(DEVICE)

print('Model:', MODEL_NAME)
print('Tokenizer vocabulary size:', len(bert_tokenizer))
print('Maximum token length:', MAX_LENGTH)


## Step 2: Tokenize the Same Dataset


In [ ]:
def tokenize_reviews(batch):
    return bert_tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH
    )


bert_train = train_data.map(tokenize_reviews, batched=True)
bert_valid = valid_data.map(tokenize_reviews, batched=True)
bert_test = test_data.map(tokenize_reviews, batched=True)

# Trainer expects the output column to be named labels
bert_train = bert_train.rename_column('label', 'labels')
bert_valid = bert_valid.rename_column('label', 'labels')
bert_test = bert_test.rename_column('label', 'labels')

# Remove original text because the model needs only IDs, masks, and labels
bert_train = bert_train.remove_columns(['text'])
bert_valid = bert_valid.remove_columns(['text'])
bert_test = bert_test.remove_columns(['text'])

# Convert to PyTorch tensors
bert_train.set_format('torch')
bert_valid.set_format('torch')
bert_test.set_format('torch')

print('Training dataset:', bert_train)
print('Example input IDs:', bert_train[0]['input_ids'][:20])


## Step 3: Define Evaluation Metrics


In [ ]:
def compute_metrics(evaluation_result):
    logits, labels = evaluation_result
    predictions = np.argmax(logits, axis=1)

    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1': f1_score(labels, predictions)
    }


## Step 4: Fine-Tune DistilBERT Using Trainer


In [ ]:
training_args = TrainingArguments(
    output_dir='distilbert_output',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    evaluation_strategy='epoch',
    save_strategy='no',
    logging_steps=25,
    report_to='none',
    seed=SEED
)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=bert_train,
    eval_dataset=bert_valid,
    compute_metrics=compute_metrics
)

trainer.train()


**Expected output format**

```text
{'loss': 0.6500, 'grad_norm': 1.2000, 'learning_rate': 2e-05, 'epoch': 1.0}
{'eval_loss': 0.3500, 'eval_accuracy': 0.8500, 'eval_f1': 0.8500, 'epoch': 1.0}
...
```
Exact values may differ.

## Step 5: Evaluate on the Validation Set


In [ ]:
validation_results = trainer.evaluate()

print('DistilBERT VALIDATION RESULTS')
for metric_name, metric_value in validation_results.items():
    print(f'{metric_name}: {metric_value:.4f}')


## Step 6: Test DistilBERT on Unseen Reviews


In [ ]:
test_output = trainer.predict(bert_test)
test_logits = test_output.predictions
bert_test_predictions = np.argmax(test_logits, axis=1)

bert_accuracy = accuracy_score(test_labels, bert_test_predictions)
bert_f1 = f1_score(test_labels, bert_test_predictions)

print('DistilBERT TEST RESULTS')
print('Accuracy:', round(bert_accuracy, 4))
print('F1-score:', round(bert_f1, 4))
print('\nDetailed classification report:')
print(classification_report(
    test_labels,
    bert_test_predictions,
    target_names=['Negative', 'Positive'],
    digits=4
))


## Step 7: Confusion Matrix for DistilBERT


In [ ]:
bert_cm = confusion_matrix(test_labels, bert_test_predictions)

plt.figure(figsize=(5, 4))
sns.heatmap(
    bert_cm,
    annot=True,
    fmt='d',
    cmap='Greens',
    xticklabels=['Negative', 'Positive'],
    yticklabels=['Negative', 'Positive']
)
plt.title('DistilBERT Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.show()


## Step 8: Predict a New Review with DistilBERT


In [ ]:
def predict_with_distilbert(review):
    encoded = bert_tokenizer(
        [review],
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH,
        return_tensors='pt'
    ).to(DEVICE)

    bert_model.eval()
    with torch.no_grad():
        outputs = bert_model(**encoded)
        probabilities = torch.softmax(outputs.logits, dim=1)[0].cpu().numpy()

    predicted_class = int(np.argmax(probabilities))
    predicted_name = 'Positive' if predicted_class == 1 else 'Negative'
    return predicted_name, float(probabilities[1])


prediction, positive_probability = predict_with_distilbert(new_review)

print('Review:', new_review)
print('Predicted class:', prediction)
print('Positive probability:', round(positive_probability, 4))


## Step 9: Save the Fine-Tuned DistilBERT Model


In [ ]:
bert_model.save_pretrained('saved_models/distilbert_sentiment_model')
bert_tokenizer.save_pretrained('saved_models/distilbert_sentiment_model')

print('Fine-tuned DistilBERT model saved in saved_models/distilbert_sentiment_model')


---
# FINAL OUTPUT AND COMPARISON


In [ ]:
final_results = pd.DataFrame({
    'Model': ['Word2Vec + LSTM', 'Fine-tuned DistilBERT'],
    'Type': ['Recurrent Neural Network', 'Pretrained Transformer'],
    'Test Accuracy': [f'{lstm_accuracy:.4f}', f'{bert_accuracy:.4f}'],
    'Test F1-score': [f'{lstm_f1:.4f}', f'{bert_f1:.4f}']
})

print('FINAL COMPARISON')
print(final_results.to_string(index=False))


## Simple Conclusion

1. In **Question 1**, Word2Vec learned word vectors from the IMDb training reviews, and those vectors were passed to an LSTM for sentiment classification.
2. In **Question 2**, DistilBERT was already pretrained on a very large text corpus. Fine-tuning adjusted it for the positive/negative movie-review task.
3. DistilBERT normally gives stronger accuracy because it understands the complete context of a sentence using self-attention.
4. Word2Vec + LSTM is easier to understand and requires fewer resources, while DistilBERT generally provides better performance.
